In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType,TimestampType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproysmartdata01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/SMS_Plataforma_Nueva.csv"

In [0]:
sms_nuevo_schema = StructType(fields=[
    StructField("cliente", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("fecha_envio", TimestampType(), True),
    StructField("usuario_creacion", StringType(), True),
    StructField("control", StringType(), True),
    StructField("telefono_normalizado", StringType(), True)
])

In [0]:
df_sms_nuevo = spark.read\
.option('header', True)\
.schema(sms_nuevo_schema)\
.csv(ruta)

In [0]:
sms_nuevo_final_df = df_sms_nuevo.select(
    col("cliente"),
    col("producto"),
    col("fecha_envio"),
    col("usuario_creacion"),
    col("control"),
    col("telefono_normalizado")
).withColumn("ingestion_date", current_timestamp())


In [0]:
sms_nuevo_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.sms_nuevo")